# Company HR Agent with RAG

This notebook builds a simple HR assistant that answers employee policy questions using the Q&A knowledge base created from `data/questions_answer.txt`.

It follows the same pattern as the RAG notebook, but swaps in HR-specific knowledge.

In [1]:
import sys
from pathlib import Path

import chromadb
import dotenv
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

ROOT_DIR = Path.cwd().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from rag_setup.question_answer_rag_setup import setup_question_answer_rag

In [2]:
# Create the Q&A collection if it does not already exist.
chroma_client = chromadb.PersistentClient(str(ROOT_DIR / "chroma"))
collection_name = "nutrition_qna"

try:
    existing = chroma_client.get_collection(name=collection_name)
except Exception:
    setup_question_answer_rag(collection_name=collection_name)
    existing = chroma_client.get_collection(name=collection_name)

hr_db = existing

In [3]:
# Quick sanity check: preview a few relevant policy results.
results = hr_db.query(query_texts=["How do I request approval for professional development courses?"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print("--- Result", i + 1, "---")
    print(doc)
    print()

--- Result 1 ---
Question: How do I request approval for professional development courses?
Answer: Fill out the educational assistance form available on the HR intranet and get sign-off from your direct supervisor. The company reimburses up to $1,500 per year for approved career-related certifications or courses.

--- Result 2 ---
Question: Where can I find our company safety and emergency guidelines?
Answer: Detailed emergency evacuation plans and safety protocols are posted on every floor near the elevators and on the company handbook page. Safety wardens conduct bi-annual fire drills for all staff.



In [5]:
@function_tool
def hr_knowledge_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Look up HR policy answers in the employee Q&A knowledge base.

    Args:
        query: The employee question to search for.
        max_results: Number of matching answers to return.

    Returns:
        A formatted string with the most relevant HR answers.
    """
    results = hr_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No HR policy information found for: {query}"

    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        question = metadata.get("question", "Unknown question")
        answer = metadata.get("answer", "No answer available")
        formatted_results.append(f"Q: {question}\nA: {answer}")

    return "HR Knowledge Base Results:\n\n" + "\n\n".join(formatted_results)

Let's test the tool directly before creating the agent.

In [ ]:
# hr_knowledge_lookup_tool("How many PTO days do I get each year?")

In [6]:
hr_agent = Agent(
    name="Company HR Assistant",
    instructions="""
    You are a helpful company HR assistant.
    Answer employee HR policy questions clearly and concisely.
    Use the hr_knowledge_lookup_tool whenever the user asks about company policies, benefits,
    payroll, PTO, remote work, parental leave, safety, performance reviews, or workplace conduct.
    If the answer is not available in the knowledge base, say so honestly and suggest contacting HR.
    """,
    tools=[hr_knowledge_lookup_tool],
)

In [7]:
with trace("Company HR Assistant with RAG"):
    result = await Runner.run(
        hr_agent,
        "How do I request approval for professional development courses?",
    )
    print(result.final_output)

Here's how to request approval for professional development courses:

- Fill out the educational assistance form on the HR intranet.
- Get sign-off from your direct supervisor.
- The company reimburses up to $1,500 per year for approved career-related certifications or courses.

If you’d like, I can help you locate the form on the intranet or provide the approval contact.


In [8]:
with trace("Company HR Assistant with RAG"):
    result = await Runner.run(
        hr_agent,
        "What should I do if I experience or witness workplace harassment?",
    )
    print(result.final_output)

If you experience or witness harassment:

- Report the incident immediately to the HR grievance committee or use the anonymous ethics hotline.
- All reports are investigated confidentially, and there is a strict zero-tolerance policy for retaliation.
- Provide factual details (what happened, when, where, who was involved, any witnesses, and any supporting evidence).
- You will be guided through the next steps by HR; you can request assistance or accommodations if needed.

If you’d like, I can help you start a report or point you to the hotline.
